# Entrenamiento YOLO Tiny para clasificacion de granadillas\n\nEste notebook servira para entrenar un modelo YOLOv3-Tiny o YOLOv4-Tiny para detectar y clasificar granadillas.\n\nClases iniciales:\n- granadilla_buena\n- granadilla_golpeada\n- granadilla_inmadura\n

## 1. Verificar GPU\n\nAntes de entrenar, en Google Colab se debe activar GPU desde:\n\nRuntime > Change runtime type > GPU\n

In [ ]:
!nvidia-smi\n

## 2. Clonar Darknet

En esta seccion se descarga el repositorio Darknet de AlexeyAB, que se usara para entrenar YOLOv3-Tiny o YOLOv4-Tiny en Google Colab.


In [ ]:
!git clone https://github.com/AlexeyAB/darknet.git
%cd darknet


## 3. Compilar Darknet con GPU y OpenCV

Se activan las opciones de GPU, CUDNN y OpenCV en el Makefile. Esto permite entrenar usando la GPU de Colab.


In [ ]:
!sed -i 's/GPU=0/GPU=1/' Makefile
!sed -i 's/CUDNN=0/CUDNN=1/' Makefile
!sed -i 's/OPENCV=0/OPENCV=1/' Makefile
!make


## 4. Definir clases del proyecto

Las clases iniciales del sistema de granadillas seran:

- granadilla_buena
- granadilla_golpeada
- granadilla_inmadura


In [ ]:
classes = [
    "granadilla_buena",
    "granadilla_golpeada",
    "granadilla_inmadura"
]

with open("data/obj.names", "w") as f:
    for c in classes:
        f.write(c + "\n")

print("Clases guardadas en data/obj.names")
!cat data/obj.names


In [ ]:
with open("data/obj.data", "w") as f:
    f.write("classes = 3\n")
    f.write("train = data/train.txt\n")
    f.write("valid = data/valid.txt\n")
    f.write("names = data/obj.names\n")
    f.write("backup = backup/\n")

!mkdir -p backup
print("Archivo data/obj.data creado")
!cat data/obj.data


## 5. Cargar dataset

En esta seccion se cargara el dataset cuando ya se tengan las imagenes etiquetadas.

Opciones futuras:

1. Descargar desde Roboflow.
2. Montar Google Drive.
3. Subir un archivo .zip manualmente.

Por ahora esta parte queda como plantilla.


In [ ]:
# Opcion A: montar Google Drive
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
# Opcion B: descargar desde Roboflow
# Cuando tengas el dataset en Roboflow, se reemplazara este bloque
# con el codigo de descarga que Roboflow genera automaticamente.

# Ejemplo referencial:
# !pip install roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key="TU_API_KEY")
# project = rf.workspace("TU_WORKSPACE").project("TU_PROYECTO")
# dataset = project.version(1).download("darknet")


## 6. Estructura esperada del dataset

Darknet espera archivos de imagen y etiquetas en formato YOLO.

Cada imagen debe tener un archivo .txt con el mismo nombre.

Ejemplo:

imagen_001.jpg  
imagen_001.txt  

Formato de etiqueta YOLO:

class_id x_center y_center width height

Los valores deben estar normalizados entre 0 y 1.


## 7. Preparar configuracion YOLOv3-Tiny

Para entrenar YOLO con 3 clases, se deben ajustar los parametros `classes` y `filters` en el archivo .cfg.

Formula para filtros:

filters = (clases + 5) * 3

Para 3 clases:

filters = (3 + 5) * 3 = 24


In [ ]:
# Crear una copia del cfg base de YOLOv3-Tiny
!cp cfg/yolov3-tiny.cfg cfg/yolov3-tiny-granadilla.cfg

# Ajustes principales para 3 clases
!sed -i 's/classes=80/classes=3/g' cfg/yolov3-tiny-granadilla.cfg
!sed -i 's/filters=255/filters=24/g' cfg/yolov3-tiny-granadilla.cfg

# Ajustes iniciales de entrenamiento
!sed -i 's/batch=1/batch=64/g' cfg/yolov3-tiny-granadilla.cfg
!sed -i 's/subdivisions=1/subdivisions=16/g' cfg/yolov3-tiny-granadilla.cfg
!sed -i 's/max_batches = 500200/max_batches = 6000/g' cfg/yolov3-tiny-granadilla.cfg
!sed -i 's/steps=400000,450000/steps=4800,5400/g' cfg/yolov3-tiny-granadilla.cfg

print("Configuracion YOLOv3-Tiny personalizada creada")


## 8. Preparar configuracion YOLOv4-Tiny

Tambien se deja preparada una configuracion alternativa para YOLOv4-Tiny.


In [ ]:
!cp cfg/yolov4-tiny.cfg cfg/yolov4-tiny-granadilla.cfg

!sed -i 's/classes=80/classes=3/g' cfg/yolov4-tiny-granadilla.cfg
!sed -i 's/filters=255/filters=24/g' cfg/yolov4-tiny-granadilla.cfg

!sed -i 's/batch=1/batch=64/g' cfg/yolov4-tiny-granadilla.cfg
!sed -i 's/subdivisions=1/subdivisions=16/g' cfg/yolov4-tiny-granadilla.cfg
!sed -i 's/max_batches = 500500/max_batches = 6000/g' cfg/yolov4-tiny-granadilla.cfg
!sed -i 's/steps=400000,450000/steps=4800,5400/g' cfg/yolov4-tiny-granadilla.cfg

print("Configuracion YOLOv4-Tiny personalizada creada")


## 9. Descargar pesos preentrenados

Antes de entrenar desde cero, se descargan pesos convolucionales preentrenados.

Esto ayuda a que el entrenamiento sea mas rapido y estable.


In [ ]:
# Pesos para YOLOv3-Tiny
!wget -nc https://pjreddie.com/media/files/darknet53.conv.74

# Pesos para YOLOv4-Tiny
!wget -nc https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v4_pre/yolov4-tiny.conv.29


## 10. Entrenar YOLOv3-Tiny

Ejecutar esta celda cuando el dataset ya este listo y existan los archivos:

- data/train.txt
- data/valid.txt
- data/obj.names
- data/obj.data


In [ ]:
# Entrenamiento YOLOv3-Tiny
# !./darknet detector train data/obj.data cfg/yolov3-tiny-granadilla.cfg darknet53.conv.74 -dont_show -map


## 11. Entrenar YOLOv4-Tiny

Ejecutar esta celda si se desea entrenar el modelo alternativo YOLOv4-Tiny.


In [ ]:
# Entrenamiento YOLOv4-Tiny
# !./darknet detector train data/obj.data cfg/yolov4-tiny-granadilla.cfg yolov4-tiny.conv.29 -dont_show -map


## 12. Pesos finales

Los mejores pesos se guardaran en la carpeta `backup/`.

Ejemplos:

- yolov3-tiny-granadilla_best.weights
- yolov4-tiny-granadilla_best.weights

Estos archivos se copiaran luego a la Raspberry Pi para inferencia con OpenCV DNN.
